# Indy framework3.0 gRPC examples

## Config server

In [1]:
from interfaces.config_socket_client import ConfigSocketClient

ip = '192.168.6.146'
config = ConfigSocketClient(ip)

In [3]:
config.GetCollSensLevel()

{'level': 5}

In [4]:
config.GetDIConfigList()

{'di_configs': [{'function_code': 65,
   'signals': [{'state': 1, 'address': 0}, {'address': 2, 'state': 0}]}]}

In [4]:
config.GetSafetyLimits()

{'power_limit': 1500.0,
 'power_limit_ratio': 100.0,
 'tcp_force_limit': 800.0,
 'tcp_force_limit_ratio': 100.0,
 'tcp_speed_limit': 4.0,
 'tcp_speed_limit_ratio': 100.0,
 'joint_limits': [175.0, 175.0, 175.0, 175.0, 175.0, 215.0]}

In [9]:
config.GetMountPos()

{'ry': 0.0, 'rz': 0.0}

## Create gRPC client objects

In [5]:
import signal
import grpc
from time import sleep
from concurrent import futures

import common as Common
import interfaces as SocketInf

In [6]:
from interfaces.control_socket_client import ControlSocketClient
from interfaces.config_socket_client import ConfigSocketClient

ip = '192.168.6.146'

control = ControlSocketClient(ip)
config = ConfigSocketClient(ip)

## Control - Custom variable

In [70]:
control.GetVariableNameList()

[]

In [59]:
int_vars_to_set = [
    {"addr": 0, "value": 10},
    {"addr": 1, "value": 20},
    {"addr": 10, "value": 30},
    {"addr": 22, "value": 40}
]
control.SetIntVariable(int_vars_to_set)

{}

In [23]:
int_vars_to_set = [    
    {"addr": 300, "value": 1}
]
control.SetIntVariable(int_vars_to_set)

{}

In [24]:
int_vars_to_set = [    
    {"addr": 300, "value": 2}
]
control.SetIntVariable(int_vars_to_set)

{}

In [63]:
data = control.GetIntVariable()
value = None
for item in data:
    if item['addr'] == 33:
        value = item['value']
        break

In [66]:
value

In [84]:
control.GetIntVariable()

[]

In [22]:
control.GetIntVariable()

[{'value': 10, 'addr': 0},
 {'addr': 1, 'value': 20},
 {'addr': 10, 'value': 80},
 {'addr': 22, 'value': 40}]

In [18]:
control.GetVariableNameList()

GRPC Exception at GetVariableNameList - <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.INTERNAL
	details = "Exception deserializing response!"
	debug_error_string = "None"
>


In [2]:
import time
import grpc
import common
import robot

control_channel = grpc.insecure_channel("{}:{}".format('192.168.6.146', 20132))
control_stub = robot.ControlRPC.ControlStub(control_channel)
control_client = robot.RobotControlClient(control_stub)


AttributeError: module 'robot' has no attribute 'ControlRPC'

## Get Joint Pos

In [14]:
control_data = control_client.get_control_data()
print("Joint angle: {:.4f}, {:.4f}, {:.4f}, {:.4f}, {:.4f}, {:.4f}".format(*list(control_data.q) ))
print("Joint vel: {:.4f}, {:.4f}, {:.4f}, {:.4f}, {:.4f}, {:.4f}".format(*list(control_data.qdot) ))

Joint angle: 30.6058, -6.8474, -3.1525, -0.9341, 24.6467, -1.4826
Joint vel: -0.0000, -0.0000, 0.0000, -0.0000, -0.0000, -0.0000


## Joint move

```
jpos = [deg, deg, deg, deg, deg, deg]
vel_ratio : 0 - 100 (%)    
```

In [27]:
control_client.move_stop()

In [ ]:
target_pos = [30, 30, 30, 30, 30, 30]
vel_ratio = 10
control_client.movej(jpos=target_pos, blending_type=common.common_msgs.BLENDING_TYPE_OVERRIDE, vel_ratio=vel_ratio)


## 절대 무브 (MoveTo)
control_client.movej(jpos=target_pos, 
                     base_type=common.common_msgs.JOINT_BASE_TYPE_ABSOLUTE, 
                     blending_type=common.common_msgs.BLENDING_TYPE_OVERRIDE, 
                     vel_ratio=vel_ratio)

## 상대 뭅즈 (MoveBy)
control_client.movej(jpos=target_pos, 
                     base_type=common.common_msgs.JOINT_BASE_TYPE_RELATIVE, 
                     blending_type=common.common_msgs.BLENDING_TYPE_OVERRIDE, 
                     vel_ratio=vel_ratio)


In [ ]:
## 0도 다갔다가 30도로 감
target1 = [0,0,0,0,0,0]
target2 = [30,0,0,0,0,0]
vel_ratio = 10

control_client.movej(jpos=target1, 
                     base_type=common.common_msgs.JOINT_BASE_TYPE_ABSOLUTE, 
                     blending_type=common.common_msgs.BLENDING_TYPE_NONE, 
                     vel_ratio=vel_ratio)

control_client.movej(jpos=target2, 
                     base_type=common.common_msgs.JOINT_BASE_TYPE_ABSOLUTE, 
                     blending_type=common.common_msgs.BLENDING_TYPE_NONE, 
                     vel_ratio=vel_ratio)

In [ ]:
## 0도 가는 중간에 멈추고 다갔다가 30도로 감
import time
target1 = [0,0,0,0,0,0]
target2 = [30,0,0,0,0,0]
vel_ratio = 10

control_client.movej(jpos=target1, 
                     base_type=common.common_msgs.JOINT_BASE_TYPE_ABSOLUTE, 
                     blending_type=common.common_msgs.BLENDING_TYPE_OVERRIDE, 
                     vel_ratio=vel_ratio)
time.sleep(1)
control_client.movej(jpos=target2, 
                     base_type=common.common_msgs.JOINT_BASE_TYPE_ABSOLUTE, 
                     blending_type=common.common_msgs.BLENDING_TYPE_OVERRIDE, 
                     vel_ratio=vel_ratio)

In [34]:
target_pos = [30, 30, 30, 30, 30, 30]
vel_ratio = 10
control_client.movej(jpos=target_pos, blending_type=common.common_msgs.BLENDING_TYPE_OVERRIDE, vel_ratio=vel_ratio)

In [33]:
target_pos = [0, 0, 0, 0, 0, 0]
vel_ratio = 10
control_client.movej(jpos=target_pos, blending_type=common.common_msgs.BLENDING_TYPE_OVERRIDE, vel_ratio=vel_ratio)

In [32]:
target_pos = [30, 30, 30, 30, 30, 30]
vel_ratio = 10
control_client.movej(jpos=target_pos, blending_type=common.common_msgs.BLENDING_TYPE_NONE, vel_ratio=vel_ratio)

In [35]:
target_pos = [0, 0, 0, 0, 0, 0]
vel_ratio = 10
control_client.movej(jpos=target_pos, blending_type=common.common_msgs.BLENDING_TYPE_NONE, vel_ratio=vel_ratio)

## Emergency related

In [ ]:
control_client.set_servo()

In [ ]:
control_client.set_brake()

In [1]:
import logging
logging.basicConfig()
log = logging.getLogger()
log.setLevel(logging.DEBUG)